<a href="https://colab.research.google.com/github/Lobnaait/SEARCH_Lobna_Tsetline_CMRI/blob/main/Tsetline_ACDCdataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from huggingface_hub import snapshot_download
data_dir = snapshot_download(repo_id="mathpluscode/ACDC", allow_patterns=["*.nii.gz", "*.csv"], repo_type="dataset")
# https://huggingface.co/datasets/mathpluscode/ACDC

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 752 files:   0%|          | 0/752 [00:00<?, ?it/s]

In [2]:
!ls /root/.cache/huggingface/hub/datasets--mathpluscode--ACDC/snapshots/4a312b610f395eda5dc0aa36f087dd6ca9991e41/

test  test.csv	train  train.csv


In [3]:
!pip install pyTsetlinMachine

  Preparing metadata (setup.py) ... done
  Created wheel for pyTsetlinMachine: filename=pytsetlinmachine-0.6.6-cp313-cp313-linux_x86_64.whl size=59793 sha256=15e8b1c78cf038d3380acb48b8a3fa3d7f9e2910ad97130a21bc15d860bad1a7
  Stored in directory: /root/.cache/pip/wheels/a9/66/98/535c2cc844fdb6fc12f31feefbaa6a33222b1378914098f498
Successfully built pyTsetlinMachine


In [ ]:
#IMPORT LIBRARIES
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, auc, roc_auc_score, classification_report, confusion_matrix
from pyTsetlinMachine.tm import MultiClassTsetlinMachine

In [ ]:
#EXTRACT DATASET
import os

train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

print(train.shape)
print(test.shape)

train.head(10)

In [ ]:
TARGET = "pathology"
print(train[TARGET].value_counts())


In [ ]:
# EXTRACT TRAININNG SET AND VALIDATION SET
print(train.columns.tolist())
features = train.columns.tolist()
features.remove(TARGET)
features.remove('pid')
print(features)

print("\n ")
X = train[features].copy()
y = train[TARGET].copy()

print("X_train shape:", X.shape)
print("y_train shape:", y.shape)
print("\nClasses:", y.value_counts())

In [ ]:
# EXTRACT TEST SET
features = test.columns.tolist()
features.remove(TARGET)
features.remove('pid')

X_test = test[features].copy()
y_test = test[TARGET].copy()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("\nClasses:", y_test.value_counts())


In [ ]:
def compute_thresholds(X, n_thresholds):
  percentile_values = np.linspace(0, 100, int(n_thresholds) + 2)[1:-1] # Exclude 0 and 100
  thresholds = []
  for feature_index in range(X.shape[1]):
      feature_thresholds = np.percentile(X[:, feature_index], percentile_values)
      feature_thresholds = np.unique(feature_thresholds) # Remove duplicate thresholds
      thresholds.append(feature_thresholds)
  return thresholds

In [ ]:
# ---------------------------------------------------------
# Convert continuous features into binary features
# ---------------------------------------------------------
import numpy as np

def threshold_binarize(X, thresholds):
    binary_features = []

    for feature_index in range(X.shape[1]):
        for threshold in thresholds[feature_index]:
            binary_features.append( (X[:, feature_index] >= threshold).astype(np.uint32) )

    return np.array(binary_features).T

In [ ]:
# Reproducible split
X_train, X_validation, y_train, y_validation = train_test_split(X, y, test_size=0.25, stratify=y)

# Convert pandas dataframes to NumPy
X_train_np = X_train.to_numpy(dtype=np.float32)
X_validation_np = X_validation.to_numpy(dtype=np.float32)
X_test_np = X_test.to_numpy(dtype=np.float32)

# Encode class labels as integers
label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_validation_enc = label_encoder.transform(y_validation)
y_test_enc = label_encoder.transform(y_test)

In [ ]:
# ---------------------------------------------------------
# Find optimal number of thresholds
# ---------------------------------------------------------

threshold_candidates = range(1, 11)

results = []

for n_thresholds in threshold_candidates:
    #print("=" * 60)
    #print(f"Testing n_thresholds = {n_thresholds}")

    # IMPORTANT:
    # thresholds are calculated ONLY from training data
    thresholds = compute_thresholds(X_train_np, n_thresholds)

    # Apply exactly the same thresholds to train and validation
    X_train_bin = threshold_binarize( X_train_np, thresholds)
    X_validation_bin = threshold_binarize( X_validation_np, thresholds)

    # Tsetlin Machine
    tm = MultiClassTsetlinMachine(number_of_clauses=1000, T=50, s=5.0)
    tm.fit(X_train_bin, y_train_enc, epochs=100)
    predictions = tm.predict(X_validation_bin)

    accuracy = accuracy_score(y_validation_enc, predictions)
    macro_f1 = f1_score(y_validation_enc,predictions,average="macro")

    #results.append(accuracy)
    results.append({
        "n_thresholds": n_thresholds,
        "binary_features": X_train_bin.shape[1],
        "accuracy": accuracy,
        "macro_f1": macro_f1,
    })
    print( f"Thresholds: {n_thresholds:2d} | " f"accuracy: {accuracy:.4f} |" f"Macro F1: {macro_f1:.4f}")



In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="macro_f1",
    ascending=False
).reset_index(drop=True)

print(results_df)

best_n_thresholds = int(
    results_df.loc[
        results_df["macro_f1"].idxmax(),
        "n_thresholds"
    ]
)

print(
    "Optimal number of thresholds:",
    best_n_thresholds
)


In [ ]:
import matplotlib.pyplot as plt

results_sorted = results_df.sort_values("n_thresholds")

plt.figure(figsize=(9, 5))

plt.plot(
    results_sorted["n_thresholds"],
    results_sorted["macro_f1"],
    marker="o",
    label="Macro F1"
)

plt.plot(
    results_sorted["n_thresholds"],
    results_sorted["accuracy"],
    marker="s",
    label="Accuracy"
)

plt.xlabel("Number of Thresholds per Feature")
plt.ylabel("Score")
plt.title("Tsetlin Machine: Threshold Optimization")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
X_train_full = np.vstack([ X_train_np,X_validation_np])
y_train_full = np.concatenate([y_train_enc,y_validation_enc])

# Recalculate thresholds using ALL training data
final_thresholds = compute_thresholds( X_train_full, best_n_thresholds)
X_train_full_bin = threshold_binarize(X_train_full, final_thresholds)
X_test_bin = threshold_binarize(X_test_np, final_thresholds)

print("Final training shape:", X_train_full_bin.shape)
print("Final test shape:", X_test_bin.shape)

final_tm = MultiClassTsetlinMachine(1000,50,5.0)

final_tm.fit( X_train_full_bin, y_train_full, epochs=100)

test_predictions = final_tm.predict(X_test_bin)

In [ ]:
print("Test Accuracy:",accuracy_score(y_test_enc, test_predictions))
print("Test Macro F1:", f1_score( y_test_enc, test_predictions, average="macro"))
print("\nClassification Report:")
print(classification_report(y_test_enc,test_predictions,target_names=label_encoder.classes_ ))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_enc,test_predictions))